In [2]:
from __future__ import annotations

import os, sys
from pathlib import *
from typing import Dict, Iterable, Optional, Tuple, List
import math
import pandas as pd
import pysam


In [3]:
root0 = Path("/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/VCF")
root_vcf = root0 / 'parkinson/ukb-b-16943-siblings'
files = os.listdir(root_vcf)
files

['ukb-b-16943_raw.vcf.gz',
 'ukb-b-16943.vcf.gz',
 'ukb-b-16943_raw.vcf.gz.tbi',
 'README.md',
 'ukb-b-16943.vcf.gz.tbi']

In [ ]:
def gt_to_dosage(gt: Optional[Tuple[Optional[int], ...]]) -> Optional[int]:
    """
    Convert a pysam GT tuple to allele dosage for diploid biallelic variants.

    Examples:
        (0, 0) -> 0
        (0, 1) or (1, 0) -> 1
        (1, 1) -> 2
        (None, None) -> None

    Returns None for missing or unsupported genotypes.
    """
    if gt is None:
        return None

    # Remove phasing notion; pysam GT is usually tuple like (0, 1)
    if len(gt) != 2:
        return None

    a1, a2 = gt
    if a1 is None or a2 is None:
        return None

    # Only allele 0/1 supported here
    if a1 not in (0, 1) or a2 not in (0, 1):
        return None

    return a1 + a2


def variant_id_from_record(rec) -> str:
    """
    Build a stable variant ID.
    Example: chr1:12345:A:G
    """
    alt = rec.alts[0] if rec.alts else "."
    return f"{rec.chrom}:{rec.pos}:{rec.ref}:{alt}"


def vcf_to_gwas_matrix(fname_vcf: str, phenotype_map: Dict[str, int], 
                       min_call_rate: float = 0.95, min_maf: float = 0.01, 
                       require_pass: bool = False,) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.DataFrame]:
    """
    Convert a VCF/BCF into:
      df         genotype matrix (samples x variants)
      y         phenotype vector (samples)
      df_ctrl    control-only matrix
      df_case    case-only matrix

    phenotype_map:
      dict like {"sample1": 0, "sample2": 1, ...}
      where 0=normal/control and 1=disease/case

    Filters:
      - keeps only biallelic SNPs
      - optional FILTER=PASS
      - min call rate
      - min minor allele frequency (MAF)

    Missing genotypes are imputed with the variant mean dosage.
    """

    filename = str(root_vcf / fname_vcf)
    vcf = pysam.VariantFile(filename)

    samples = list(vcf.header.samples)
    samples_kept = [s for s in samples if s in phenotype_map]

    if not samples_kept:
        raise ValueError("No VCF samples matched phenotype_map.")

    variant_columns: List[str] = []
    matrix_data: List[List[float]] = []

    for rec in vcf.fetch():
        # Optional FILTER=PASS
        if require_pass and len(list(rec.filter.keys())) > 0 and "PASS" not in rec.filter.keys():
            continue

        # Keep only biallelic SNPs
        if rec.alts is None or len(rec.alts) != 1:
            continue
        if len(rec.ref) != 1 or len(rec.alts[0]) != 1:
            continue

        dosages = []
        n_called = 0
        alt_alleles = 0

        for sample in samples_kept:
            gt = rec.samples[sample].get("GT")
            dosage = gt_to_dosage(gt)
            dosages.append(dosage)

            if dosage is not None:
                n_called += 1
                alt_alleles += dosage

        # Call rate
        call_rate = n_called / len(samples_kept)
        if call_rate < min_call_rate:
            continue

        # MAF
        if n_called == 0:
            continue
        af = alt_alleles / (2 * n_called)
        maf = min(af, 1 - af)
        if maf < min_maf:
            continue

        # Mean imputation for missing calls
        mean_dosage = alt_alleles / n_called
        dosages = [mean_dosage if d is None else d for d in dosages]

        variant_columns.append(variant_id_from_record(rec))
        matrix_data.append(dosages)

    if not matrix_data:
        raise ValueError("No variants passed filtering.")

    # matrix_data currently = variants x samples
    df = pd.DataFrame(matrix_data, index=variant_columns, columns=samples_kept).T
    y = pd.Series({s: phenotype_map[s] for s in samples_kept}, name="phenotype")

    df_ctrl = df.loc[y[y == 0].index].copy()
    df_case = df.loc[y[y == 1].index].copy()

    return df, y, df_ctrl, df_case